In [53]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

import shutil


from torch.utils.data import Dataset
from PIL import Image
import torch


from collections import defaultdict
from pathlib import Path

from torchvision import transforms

from PIL import Image
import os

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights

from collections import Counter
import hashlib

from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

from sklearn.metrics import (
    classification_report,
    f1_score
)

In [54]:
def create_video_groups(paths):

    groups = defaultdict(list)

    for path in paths:

        path = Path(path)

        # images/testing/flip/0001_000000010.jpg
        label = path.parent.name             # "flip"
        split = path.parent.parent.name       # "testing"

        group_id, frame_id = path.stem.split("_")

        group_id = int(group_id)
        frame_id = int(frame_id)

        # This uniquely identifies ONE source video
        video_key = (split, label, group_id)

        groups[video_key].append(
            (frame_id, path)
        )

    # Put frames in chronological order INSIDE each video
    for video_key in groups:

        groups[video_key].sort(
            key=lambda x: x[0]
        )

    return groups

def make_sequences(groups, sequence_length=16):

    sequences = []
    labels = []
    video_ids = []

    label_map = {
        "notflip": 0,
        "flip": 1
    }

    for video_key, frames in groups.items():

        split, label, group_id = video_key

        # Remove frame numbers now that they're sorted
        ordered_paths = [
            path for frame_id, path in frames
        ]

        # Make 16-frame windows INSIDE this video
        for start in range(
            0,
            len(ordered_paths) - sequence_length + 1,
            sequence_length
        ):

            sequence = ordered_paths[
                start:start + sequence_length
            ]

            sequences.append(sequence)

            labels.append(
                label_map[label]
            )

            # Keep track of source video
            video_ids.append(video_key)

    return sequences, labels, video_ids


def save_sequence_to_folder(sequence, output_folder):

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    for i, (path, label) in enumerate(sequence):

        path = Path(path)

        new_filename = f"frame_{i + 1:03d}_{label}{path.suffix}"

        destination = output_folder / new_filename

        shutil.copy2(path, destination)
    

In [55]:
class FlipCNNLSTM(nn.Module):

    def __init__(
        self,
        cnn,
        hidden_size=128,
        num_layers=1,
        num_classes=2
    ):
        super().__init__()

        self.cnn = cnn

        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )


    def forward(self, x):

        batch_size, seq_len, channels, height, width = x.shape

        # Treat all frames temporarily as individual images
        x = x.reshape(
            batch_size * seq_len,
            channels,
            height,
            width
        )

        # Extract CNN features
        features = self.cnn(x)

        # Restore temporal structure
        features = features.reshape(
            batch_size,
            seq_len,
            512
        )

        # Process sequence
        output, (hidden, cell) = self.lstm(features)

        final_hidden = hidden[-1]

        logits = self.classifier(final_hidden)

        return logits




class FrameSequenceDataset(Dataset):

    def __init__(self, sequences, labels, transform=None):
        self.sequences = sequences
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):

        sequence_paths = self.sequences[index]

        frames = []

        for path in sequence_paths:

            image = Image.open(path).convert("RGB")

            if self.transform:
                image = self.transform(image)

            frames.append(image)

        sequence = torch.stack(frames)

        label = self.labels[index]

        return sequence, label

In [56]:
main_folder = Path("images")

all_paths = list(
    main_folder.rglob("*.jpg")
)

video_groups = create_video_groups(
    all_paths
)

sequences, labels, video_ids = make_sequences(
    video_groups,
    sequence_length=16
)

print(len(sequences))
print(len(labels))

print(labels[:90])
print(video_ids[:90])

weights = ResNet18_Weights.DEFAULT

transform = weights.transforms()

train_sequences, test_sequences = train_test_split(
    sequences,
    test_size=0.2,
    random_state=123
)

93
93
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[('training', 'flip', 5), ('training', 'flip', 38), ('training', 'flip', 38), ('training', 'flip', 11), ('training', 'flip', 62), ('training', 'flip', 10), ('training', 'flip', 39), ('training', 'flip', 4), ('training', 'flip', 60), ('training', 'flip', 60), ('training', 'flip', 13), ('training', 'flip', 7), ('training', 'flip', 48), ('training', 'flip', 6), ('training', 'flip', 12), ('training', 'flip', 15), ('training', 'flip', 28), ('training', 'flip', 29), ('training', 'flip', 17), ('training', 'flip', 59), ('training', 'flip', 3), ('training', 'flip', 65), ('training', 'flip', 16), ('training', 'flip', 58), ('training', 'flip', 61), ('training', 'flip', 22), ('training', 'flip', 45), ('training', 'flip', 44),

In [57]:
model = FlipLSTM(
    input_size=64 * 64 * 3,
    hidden_size=128,
    num_layers=1,
    num_classes=2
)

In [58]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

# Make each video ID one simple string
group_names = [
    f"{split}_{label}_{group_id}"
    for split, label, group_id in video_ids
]

# Take one of the 5 folds as the test set
train_idx, test_idx = next(
    sgkf.split(
        sequences,
        labels,
        groups=group_names
    )
)

train_sequences = [sequences[i] for i in train_idx]
train_labels = [labels[i] for i in train_idx]

test_sequences = [sequences[i] for i in test_idx]
test_labels = [labels[i] for i in test_idx]

In [59]:
train_dataset = FrameSequenceDataset(
    train_sequences,
    train_labels,
    transform=transform
)

test_dataset = FrameSequenceDataset(
    test_sequences,
    test_labels,
    transform=transform
)

In [60]:
sequence, label = train_dataset[0]

print(sequence.shape)
print(label)

torch.Size([16, 3, 224, 224])
1


In [61]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

In [62]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)


resnet = resnet18(weights=None)

resnet.fc = nn.Linear(
    resnet.fc.in_features,
    2
)

resnet.load_state_dict(
    torch.load(
        "resnet18_flip_classifier.pth",
        map_location=device
    )
)

resnet.fc = nn.Identity()


for param in resnet.parameters():
    param.requires_grad = False

model = FlipCNNLSTM(
    cnn=resnet,
    hidden_size=128,
    num_layers=1,
    num_classes=2
)



model = model.to(device)

print(device)


mps


In [63]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=0.001
)

In [64]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for sequences_batch, labels_batch in train_loader:

        sequences_batch = sequences_batch.to(device)
        labels_batch = labels_batch.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass through LSTM
        outputs = model(sequences_batch)

        # Calculate loss
        loss = criterion(
            outputs,
            labels_batch
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == labels_batch
        ).sum().item()

        total += labels_batch.size(0)

    train_loss = running_loss / len(train_loader)
    train_accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Loss: {train_loss:.4f} | "
        f"Accuracy: {train_accuracy:.4f}"
    )

Epoch 1/10 | Loss: 0.2258 | Accuracy: 0.9324
Epoch 2/10 | Loss: 0.0699 | Accuracy: 0.9865
Epoch 3/10 | Loss: 0.0504 | Accuracy: 0.9730
Epoch 4/10 | Loss: 0.0618 | Accuracy: 0.9730
Epoch 5/10 | Loss: 0.0364 | Accuracy: 0.9865
Epoch 6/10 | Loss: 0.0449 | Accuracy: 1.0000
Epoch 7/10 | Loss: 0.1522 | Accuracy: 0.9595
Epoch 8/10 | Loss: 0.0708 | Accuracy: 0.9865
Epoch 9/10 | Loss: 0.1286 | Accuracy: 0.9189
Epoch 10/10 | Loss: 0.1336 | Accuracy: 0.9865


In [65]:
model.eval()

correct = 0
total = 0

all_predictions = []
all_labels = []

with torch.no_grad():

    for sequences_batch, labels_batch in test_loader:

        sequences_batch = sequences_batch.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model(sequences_batch)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == labels_batch
        ).sum().item()

        total += labels_batch.size(0)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels_batch.cpu().numpy()
        )

test_accuracy = correct / total

print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 1.0000


In [66]:
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["notflip", "flip"]
    )
)

              precision    recall  f1-score   support

     notflip       1.00      1.00      1.00        10
        flip       1.00      1.00      1.00         9

    accuracy                           1.00        19
   macro avg       1.00      1.00      1.00        19
weighted avg       1.00      1.00      1.00        19

